In [1]:
import sys
from pathlib import Path
from typing import Any

import logging
from transformers.utils import logging as hf_logging
from huggingface_hub.utils import disable_progress_bars
import yaml
from langchain.chat_models import init_chat_model

NOTEBOOK_CWD = Path.cwd()
PROJECT_ROOT = NOTEBOOK_CWD if (NOTEBOOK_CWD / "data").exists() else NOTEBOOK_CWD.parent
if PROJECT_ROOT != NOTEBOOK_CWD:
    sys.path = [
        path_entry for path_entry in sys.path
        if path_entry not in ("", str(NOTEBOOK_CWD))
    ]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from graph.graph import build_train_graph
from graph.graph_state import llm_prompt, entity_output_parser, entity_schema

# 关闭 transformers 的 warning/load report
hf_logging.set_verbosity_error()
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("transformers.utils.loading_report").setLevel(logging.ERROR)

# 关闭 Loading weights / 下载进度条
hf_logging.disable_progress_bar()
disable_progress_bars()


In [2]:
train_graph = build_train_graph()

train_path = str(PROJECT_ROOT / "data/GPT_data_source/train.txt")
valid_path = str(PROJECT_ROOT / "data/matscholar/valid.txt")
unlabeled_pool_path = str(
    PROJECT_ROOT / "data/GPT_data_source/matscholar_data/train_unlabeled.jsonl"
)

ENV_CONFIG_FILE = PROJECT_ROOT / "env_config.yaml"


In [3]:
def load_env_config(path: Path = ENV_CONFIG_FILE) -> dict[str, Any]:
    if not path.exists():
        return {}
    with path.open("r", encoding="utf-8") as f:
        return yaml.safe_load(f) or {}

ENV_CONFIG = load_env_config()


In [4]:
primary_model = init_chat_model(
    model="Doubao-Seed-2.0-pro",
    model_provider="openai",
    temperature=0,
    api_key=ENV_CONFIG["openai_api_key"],
    base_url=ENV_CONFIG["openai_base_url"],
)

judge_model = init_chat_model(
    model="deepseek-v3.2",
    model_provider="openai",
    temperature=0,
    api_key=ENV_CONFIG["openai_api_key"],
    base_url=ENV_CONFIG["openai_base_url"],
)


In [5]:
res = train_graph.invoke(
    {
        "train_path": train_path,
        "valid_path": valid_path,
        "unlabeled_pool_path": unlabeled_pool_path,
    },
    context={
        "prompt": llm_prompt,
        "output_parser": entity_output_parser,
        "ner_model": "bert_bilstm_crf",
        "entity_schema": entity_schema,
        "llm": primary_model,
        "judge_llm": judge_model,
        "trace_enabled": True
    },
)

[GraphTrace] -> initialize | iter=1/6 | batch=0
[GraphTrace] <- initialize | updates=['iteration', 'distance_ratio_threshold', 'min_distance_ratio_threshold', 'max_distance_ratio_threshold', 'model_distance_ratio_threshold', 'threshold_step', 'processed_sample_ids', 'previous_metrics', 'best_metrics']
[GraphTrace] -> train_ner | iter=1/6 | batch=0
开始第1轮训练：




Epoch 1/25 - 阶段1（联合训练）
Train Loss: 22.4043
Valid Loss: 8.6945
Valid Precision: 0.6331
Valid Recall: 0.7266
Valid F1: 0.6766
保存当前最优模型: E:\大学学习\毕设\26毕设\NRE_project\graph\model\bert_bilstm_crf\bert_bilstm_crf_iter_0.pt

Epoch 2/25 - 阶段1（联合训练）
Train Loss: 5.1613
Valid Loss: 7.7961
Valid Precision: 0.6834
Valid Recall: 0.7805
Valid F1: 0.7287
保存当前最优模型: E:\大学学习\毕设\26毕设\NRE_project\graph\model\bert_bilstm_crf\bert_bilstm_crf_iter_0.pt

Epoch 3/25 - 阶段1（联合训练）
Train Loss: 2.3970
Valid Loss: 8.3192
Valid Precision: 0.7103
Valid Recall: 0.7662
Valid F1: 0.7372
保存当前最优模型: E:\大学学习\毕设\26毕设\NRE_project\graph\model\bert_bilstm_crf\bert_bilstm_crf